In [0]:
%sql
CREATE CATALOG IF NOT EXISTS atmos
MANAGED LOCATION 'abfss://atmos-landing-dev-001@saatmosdevwestus2riuler.dfs.core.windows.net/';
CREATE SCHEMA IF NOT EXISTS atmos.silver;

In [0]:
dbutils.widgets.text("ingestion_date","","Data de Processamento (YYYY-MM-DD)")
ingestion_date = dbutils.widgets.get("ingestion_date")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, DoubleType

In [0]:
df_inmet = (
    spark.table("atmos.silver.climate_inmet")
    .filter(F.col("data_ingestao")==ingestion_date)
)

df_visual_crossing = (

    spark.table("atmos.silver.climate_visual_crossing")
    .filter(F.col("data_ingestao")==ingestion_date)
)

In [0]:
df_unified = df_inmet.unionByName(df_visual_crossing)

In [0]:
(
    df_unified.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"data_ingestao = '{ingestion_date}'")
    .partitionBy("data_ingestao")
    .saveAsTable("atmos.silver.climate_unified")
)

In [0]:
count = (
    spark.table("atmos.silver.climate_unified")
    .filter(F.col("data_ingestao")==ingestion_date)
    .count()
)

assert count > 0, f"Nenhum registro gravado em silver.climate_unified para data_ingestao={ingestion_date}."
print(f"ok - {count} registros em silver.climate_unified I data_ingestion = {ingestion_date}")